# Pedestrian Accessibility Data Pipeline — Cayuga

This notebook implements a reproducible data-preparation workflow for pedestrian accessibility analysis. Road-segment and node datasets are integrated, profiled for data quality, cleaned, transformed into derived accessibility metrics, and exported for downstream Power BI and Folium visualization.

**Pipeline:** Raw CSV data → Integration → Data quality & cleaning → Feature engineering → Curated dataset → Visualization


In [22]:
import pandas as pd
import folium
from pathlib import Path


In [23]:
# Locate project directories whether Jupyter starts in the repo root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load raw source datasets
df_edges = pd.read_csv(RAW_DATA_DIR / "edges.csv")
df_nodes = pd.read_csv(RAW_DATA_DIR / "nodes.csv")

print(f"Edges: {df_edges.shape}")
print(f"Nodes: {df_nodes.shape}")

# Standardize join keys
df_edges["from"] = df_edges["from"].astype(str).str.strip()
df_nodes["osmid"] = df_nodes["osmid"].astype(str).str.strip()

df_edges = df_edges.rename(columns={"from": "id"})
df_nodes = df_nodes.rename(columns={"osmid": "id"})

# Integrate road-segment data with node attributes
df = df_edges.merge(
    df_nodes,
    how="left",
    on="id",
    suffixes=("_edge", "_node")
).rename(columns={"id": "osmid_from"})

# Normalize the original node-type field used by the project.
# In the source data this may arrive as `type`; downstream code uses `footway`.
if "type" in df.columns and "footway" not in df.columns:
    df["type"] = df["type"].fillna("sidewalk")
    df = df.rename(columns={"type": "footway"})

# The original auxiliary field is not used downstream.
if "footway_type" in df.columns:
    df = df.drop(columns=["footway_type"])

print(f"Integrated dataset: {df.shape}")
df.head()


Edges: (986, 19)
Nodes: (306, 5)
Integrated dataset: (986, 22)


,osmid_from,to,osmid,street_name,highway,lanes,maxspeed,oneway,reversed,length,...,access,crossing,crossing_markings,surface,lit,sidewalk,latitude,longitude,street_count,footway
0,416694987,6463584760,67222638,Munsee Street North,secondary,2.0,50.0,False,False,6.114355,...,NaN,no,NaN,asphalt,yes,right,42.951206,-79.856181,4,sidewalk
1,416694987,6463584763,67222638,Munsee Street North,secondary,2.0,50.0,False,True,9.588805,...,NaN,no,NaN,asphalt,yes,right,42.951206,-79.856181,4,sidewalk
2,416694987,6463584766,67226665,Kerr Street East,residential,2.0,NaN,False,True,9.446809,...,NaN,no,NaN,asphalt,no,no,42.951206,-79.856181,4,sidewalk
3,416694987,6463584750,67225880,Kerr Street West,residential,2.0,NaN,False,True,111.908025,...,NaN,no,NaN,asphalt,no,no,42.951206,-79.856181,4,sidewalk
4,416695612,4087046866,67222638,Munsee Street North,secondary,2.0,50.0,False,False,34.661426,...,NaN,no,NaN,asphalt,yes,right,42.953784,-79.857347,4,sidewalk


## Data Preprocessing

## Checking Missingness and fixing them

In [24]:
# Profile missing values before cleaning
def identify_missingness(dataframe):
    missing_count = dataframe.isnull().sum()
    missing_percent = (missing_count / len(dataframe)) * 100
    return missing_count, missing_percent

missingData, missingData_perc = identify_missingness(df)

missing_summary = pd.DataFrame({
    "Total Missing": missingData,
    "Percentage Missing": missingData_perc.round(2)
})

print(missing_summary[missing_summary["Total Missing"] > 0])


                   Total Missing  Percentage Missing
street_name                  447               45.33
lanes                        449               45.54
maxspeed                     874               88.64
bridge                       968               98.17
service                      868               88.03
access                       974               98.78
crossing_markings            906               91.89


In [25]:
# Fill missing values using the rules from the original project
def impute_missing_values(dataframe):
    dataframe = dataframe.copy()
    mean_maxspeed = dataframe["maxspeed"].mean()

    # Street name
    for i in range(len(dataframe)):
        if pd.isna(dataframe.loc[i, "street_name"]):
            if dataframe.loc[i, "street_count"] > 1:
                nearby_streets = dataframe[
                    dataframe["osmid_from"] == dataframe.loc[i, "osmid_from"]
                ]["street_name"].dropna()

                dataframe.loc[i, "street_name"] = (
                    nearby_streets.iloc[0] if not nearby_streets.empty else "Unnamed Road"
                )
            else:
                dataframe.loc[i, "street_name"] = "Unnamed Road"

    # Lane count
    for i in range(len(dataframe)):
        if pd.isna(dataframe.loc[i, "lanes"]):
            if dataframe.loc[i, "footway"] == "traffic_signals":
                dataframe.loc[i, "lanes"] = 2
            elif dataframe.loc[i, "footway"] in {"turning_circle", "stop", "crossing"}:
                dataframe.loc[i, "lanes"] = 1
            else:
                dataframe.loc[i, "lanes"] = 1

    # Maximum speed
    for i in range(len(dataframe)):
        if pd.isna(dataframe.loc[i, "maxspeed"]):
            footway = dataframe.loc[i, "footway"]

            if footway == "traffic_signals":
                dataframe.loc[i, "maxspeed"] = 40
            elif footway in {"turning_circle", "stop"}:
                dataframe.loc[i, "maxspeed"] = 30
            elif footway == "crossing":
                dataframe.loc[i, "maxspeed"] = 20
            else:
                dataframe.loc[i, "maxspeed"] = mean_maxspeed

    # Categorical fields
    dataframe["service"] = dataframe["service"].fillna("Other")
    dataframe["access"] = dataframe["access"].fillna("Other")
    dataframe["bridge"] = dataframe["bridge"].fillna("no")
    dataframe["crossing_markings"] = dataframe["crossing_markings"].fillna("no marking")

    return dataframe

df = impute_missing_values(df)


## checking the new dataframe with no missingness

In [26]:
# Validate missingness after cleaning
missingData, missingData_perc = identify_missingness(df)

missing_summary = pd.DataFrame({
    "Total Missing": missingData,
    "Percentage Missing": missingData_perc.round(2)
})

remaining_missing = missing_summary[missing_summary["Total Missing"] > 0]
print(remaining_missing if not remaining_missing.empty else "No missing values remain.")


No missing values remain.


# Feature Engineering

In [27]:
def calculate_busy_score(row):
    score = 0

    if row["lanes"] > 2:
        score += 1
    if row["maxspeed"] > 50:
        score += 1
    if row["bridge"] == "yes":
        score += 1
    if row["length"] > 500:
        score += 1
    if str(row["oneway"]).strip().lower() == "true":
        score += 1

    return score

df["busy_score"] = df.apply(calculate_busy_score, axis=1)
df["is_busy"] = df["busy_score"].apply(
    lambda x: "Busy" if x > 2 else "Not Busy"
)

print(df["is_busy"].value_counts())


is_busy
Not Busy    980
Busy          6
Name: count, dtype: int64


In [28]:
def calculate_pedestrian_friendly_score(row):
    score = 0

    if row["sidewalk"] in ["left", "right", "both"]:
        score += 2
    elif row["footway"] == "sidewalk":
        score += 1
    elif row["footway"] == "crossing":
        score += 1

    if row["surface"] in ["asphalt", "paved", "concrete"]:
        score += 1
    if row["crossing"] == "marked":
        score += 1
    if row["lit"] == "yes":
        score += 1

    return score

df["pedestrian_friendly_score"] = df.apply(
    calculate_pedestrian_friendly_score, axis=1
)

df["is_pedestrian_friendly"] = df["pedestrian_friendly_score"].apply(
    lambda x: "Pedestrian Friendly" if x > 3 else "Not Pedestrian Friendly"
)

top_pedestrian_streets = (
    df[["street_name", "pedestrian_friendly_score"]]
    .sort_values("pedestrian_friendly_score", ascending=False)
    .drop_duplicates(subset=["street_name"])
    .head(10)
)

print(df["is_pedestrian_friendly"].value_counts())
print("\nTop pedestrian-friendly streets:")
print(top_pedestrian_streets)


is_pedestrian_friendly
Not Pedestrian Friendly    958
Pedestrian Friendly         28
Name: count, dtype: int64

Top pedestrian-friendly streets:
             street_name  pedestrian_friendly_score
22   Munsee Street North                          4
467  Ottawa Street North                          3
485  Cayuga Street North                          3
832   Talbot Street East                          3
20    Norton Street West                          3
3       Kerr Street West                          2
957         Unnamed Road                          2
669    McKay Street East                          2
946   Talbot Street West                          2
52           Ouse Street                          2


In [29]:
df.head()


,osmid_from,to,osmid,street_name,highway,lanes,maxspeed,oneway,reversed,length,...,lit,sidewalk,latitude,longitude,street_count,footway,busy_score,is_busy,pedestrian_friendly_score,is_pedestrian_friendly
0,416694987,6463584760,67222638,Munsee Street North,secondary,2.0,50.000000,False,False,6.114355,...,yes,right,42.951206,-79.856181,4,sidewalk,0,Not Busy,4,Pedestrian Friendly
1,416694987,6463584763,67222638,Munsee Street North,secondary,2.0,50.000000,False,True,9.588805,...,yes,right,42.951206,-79.856181,4,sidewalk,0,Not Busy,4,Pedestrian Friendly
2,416694987,6463584766,67226665,Kerr Street East,residential,2.0,53.035714,False,True,9.446809,...,no,no,42.951206,-79.856181,4,sidewalk,1,Not Busy,2,Not Pedestrian Friendly
3,416694987,6463584750,67225880,Kerr Street West,residential,2.0,53.035714,False,True,111.908025,...,no,no,42.951206,-79.856181,4,sidewalk,1,Not Busy,2,Not Pedestrian Friendly
4,416695612,4087046866,67222638,Munsee Street North,secondary,2.0,50.000000,False,False,34.661426,...,yes,right,42.953784,-79.857347,4,sidewalk,0,Not Busy,4,Pedestrian Friendly


In [32]:
# Calculate the center of the map using the mean latitude and longitude
map_center = [df['latitude'].mean(), df['longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=14)


# Function to determine color based on pedestrian-friendly score
def get_marker_color(score):
    if score >= 3:
        return 'green'
    elif score >= 2:
        return 'yellow'
    else:
        return 'red'

# Adding markers with color coding based on pedestrian-friendly scores
for idx, row in df.iterrows():
    marker_color = get_marker_color(row['pedestrian_friendly_score'])

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        color=marker_color,
        fill=True,
        fill_color=marker_color,
        fill_opacity=0.7,
        popup=f"Pedestrian Friendly Score: {row['pedestrian_friendly_score']}"
    ).add_to(m)

# Save the map to an HTML file
m.save(OUTPUT_DIR / "pedestrian_safety_map.html")


In [33]:
# Save curated data for downstream analytics
curated_path = PROCESSED_DATA_DIR / "pedestrian_accessibility_curated.csv"
df.to_csv(curated_path, index=False)

print(f"Final shape: {df.shape}")
print(f"Remaining missing values: {int(df.isnull().sum().sum())}")
print(f"Curated dataset saved to: {curated_path}")
print(f"Map saved to: {OUTPUT_DIR / 'pedestrian_safety_map.html'}")


Final shape: (986, 26)
Remaining missing values: 0
Curated dataset saved to: f:\Cayugu\Pedestrian-Accessibility-in-Cayuga\data\processed\pedestrian_accessibility_curated.csv
Map saved to: f:\Cayugu\Pedestrian-Accessibility-in-Cayuga\outputs\pedestrian_safety_map.html
